# RLO Experiments - 8x NVIDIA B200 (HiPerGator)

## Overview

**Hardware:** 8x NVIDIA B200 (183GB VRAM each = 1.4TB total)

**Experiments (Following LION Paper Section 4):**
1. **4.1 ImageNet Classification**
   - ResNet-50: 90 epochs, batch 4096
   - ViT-S/16: 300 epochs, batch 2048
   - ViT-B/16: 300 epochs, batch 1024

2. **4.3 Diffusion Models** (CIFAR-10 for speed)

3. **4.4 Language Modeling**

**Optimizers to Compare:**
- AdamW (baseline)
- LION (Google)
- RLO (yours)
- RLO_LambdaA (yours)
- SmoothLiftedRLO (yours)

---

In [ ]:
# =============================================================================
# CELL 1: Setup and Imports
# =============================================================================

import os
import sys
import time
import math
import random
import json
import shutil
import warnings
from pathlib import Path
from datetime import datetime
from typing import Dict, List, Optional, Tuple, Any
from dataclasses import dataclass, field
from functools import partial
from concurrent.futures import ThreadPoolExecutor, as_completed
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Optimizer
from torch.utils.data import DataLoader, Dataset
import torchvision
import torchvision.transforms as T
from torchvision.datasets import ImageFolder
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['font.size'] = 12

warnings.filterwarnings('ignore')

# =============================================================================
# GPU OPTIMIZATIONS
# =============================================================================

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.set_float32_matmul_precision('high')

if hasattr(torch.backends.cuda, 'enable_flash_sdp'):
    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

# =============================================================================
# PATHS
# =============================================================================

# HuggingFace cache location (arrow format)
HF_CACHE_PATH = Path("/blue/wdixon/wang.yixuan/lypcdf/imagenet_data")

# Target ImageFolder location
IMAGENET_FOLDER = Path("/blue/wdixon/wang.yixuan/lypcdf/imagenet_folder")

# Results
RESULTS_DIR = Path("./rlo_results")
RESULTS_DIR.mkdir(exist_ok=True)

# Check GPUs
NUM_GPUS = torch.cuda.device_count()
print(f"Available GPUs: {NUM_GPUS}")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name}, {props.total_memory / 1e9:.0f}GB")

print(f"\nHF cache: {HF_CACHE_PATH}")
print(f"ImageFolder target: {IMAGENET_FOLDER}")
print(f"Results: {RESULTS_DIR}")

In [ ]:
# =============================================================================
# CELL 2: Convert HuggingFace Arrow Format to ImageFolder
# =============================================================================

def convert_hf_to_imagefolder(
    hf_cache_path: Path,
    output_path: Path,
    num_workers: int = 32,
):
    """
    Convert HuggingFace ImageNet cache to standard ImageFolder format.
    
    HF format: arrow files with image bytes + labels
    ImageFolder format:
        train/
            n01440764/
                ILSVRC2012_val_00000001.JPEG
            ...
        val/
            n01440764/
                ILSVRC2012_val_00000001.JPEG
            ...
    """
    from datasets import load_from_disk, load_dataset
    from PIL import Image
    import io
    
    output_path = Path(output_path)
    
    # Check if already converted
    train_dir = output_path / 'train'
    val_dir = output_path / 'val'
    
    if train_dir.exists() and val_dir.exists():
        # Count files
        train_count = sum(1 for _ in train_dir.rglob('*.JPEG'))
        val_count = sum(1 for _ in val_dir.rglob('*.JPEG'))
        if train_count > 1000000 and val_count > 40000:
            print(f"✓ ImageFolder already exists with {train_count} train and {val_count} val images")
            return True
    
    print("Converting HuggingFace ImageNet to ImageFolder format...")
    print("This may take 30-60 minutes...")
    
    # Load HF dataset
    print("Loading HuggingFace dataset...")
    try:
        dataset = load_dataset(
            "imagenet-1k",
            cache_dir=str(hf_cache_path),
            trust_remote_code=True,
        )
    except Exception as e:
        print(f"Error loading dataset: {e}")
        print("Trying to load from disk...")
        # Try loading directly from arrow files
        arrow_path = hf_cache_path / "imagenet-1k" / "default" / "0.0.0"
        subdirs = list(arrow_path.iterdir())
        if subdirs:
            dataset = load_from_disk(str(subdirs[0]))
        else:
            raise ValueError(f"Could not find arrow files in {arrow_path}")
    
    # Get class names (synsets)
    # ImageNet-1K has 1000 classes
    # We need to map label indices to synset IDs
    
    # Standard ImageNet synset IDs
    IMAGENET_SYNSETS = None
    try:
        # Try to get from dataset features
        if hasattr(dataset['train'].features['label'], 'names'):
            IMAGENET_SYNSETS = dataset['train'].features['label'].names
    except:
        pass
    
    if IMAGENET_SYNSETS is None:
        # Use generic class names
        IMAGENET_SYNSETS = [f"class_{i:04d}" for i in range(1000)]
    
    print(f"Found {len(IMAGENET_SYNSETS)} classes")
    
    def save_split(split_name, split_data):
        split_dir = output_path / split_name
        split_dir.mkdir(parents=True, exist_ok=True)
        
        # Create class directories
        for synset in IMAGENET_SYNSETS:
            (split_dir / synset).mkdir(exist_ok=True)
        
        print(f"Saving {split_name} split ({len(split_data)} images)...")
        
        def save_image(args):
            idx, item = args
            try:
                image = item['image']
                label = item['label']
                synset = IMAGENET_SYNSETS[label]
                
                # Convert to RGB if needed
                if image.mode != 'RGB':
                    image = image.convert('RGB')
                
                # Save
                filename = f"{split_name}_{idx:08d}.JPEG"
                filepath = split_dir / synset / filename
                image.save(filepath, 'JPEG', quality=95)
                return True
            except Exception as e:
                return False
        
        # Process in parallel
        success = 0
        failed = 0
        
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = []
            for idx in range(len(split_data)):
                futures.append(executor.submit(save_image, (idx, split_data[idx])))
            
            for future in tqdm(as_completed(futures), total=len(futures), desc=split_name):
                if future.result():
                    success += 1
                else:
                    failed += 1
        
        print(f"  {split_name}: {success} saved, {failed} failed")
        return success
    
    # Convert train and validation splits
    train_count = save_split('train', dataset['train'])
    val_count = save_split('val', dataset['validation'])
    
    print(f"\n✓ Conversion complete!")
    print(f"  Train: {train_count} images")
    print(f"  Val: {val_count} images")
    print(f"  Location: {output_path}")
    
    return True


# Check if conversion is needed
print("Checking ImageNet data...")
if (IMAGENET_FOLDER / 'train').exists() and (IMAGENET_FOLDER / 'val').exists():
    train_classes = len(list((IMAGENET_FOLDER / 'train').iterdir()))
    val_classes = len(list((IMAGENET_FOLDER / 'val').iterdir()))
    print(f"✓ ImageFolder exists: {train_classes} train classes, {val_classes} val classes")
    NEED_CONVERSION = train_classes < 1000
else:
    print("✗ ImageFolder not found, conversion needed")
    NEED_CONVERSION = True

if NEED_CONVERSION:
    print("\nTo convert, run: convert_hf_to_imagefolder(HF_CACHE_PATH, IMAGENET_FOLDER)")
    print("This will take 30-60 minutes but only needs to be done once.")

In [ ]:
# =============================================================================
# CELL 3: Run Conversion (if needed)
# =============================================================================

if NEED_CONVERSION:
    print("Starting conversion...")
    convert_hf_to_imagefolder(HF_CACHE_PATH, IMAGENET_FOLDER, num_workers=32)
else:
    print("Conversion not needed, ImageFolder already exists.")

In [ ]:
# =============================================================================
# CELL 4: All Optimizers
# =============================================================================

class RLO(Optimizer):
    """Riemannian Lyapunov Optimizer."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.1, 
                 belief_coef=0.1, eps=1e-8):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay, 
                       belief_coef=belief_coef, eps=eps)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr, wd = group["lr"], group["weight_decay"]
            beta1, beta2 = group["betas"]
            belief, eps = group["belief_coef"], group["eps"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state["exp_avg"] = torch.zeros_like(p)

                m = state["exp_avg"]
                if wd != 0.0:
                    p.mul_(1.0 - lr * wd)

                c = beta1 * m + (1.0 - beta1) * g
                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                d = c.sign() + belief * (delta / delta_norm)
                p.add_(d, alpha=-lr)
                m.mul_(beta2).add_(g, alpha=(1.0 - beta2))

        return loss


class RLO_LambdaA(Optimizer):
    """RLO with adaptive preconditioning."""
    def __init__(self, params, lr=1e-4, beta1=0.9, beta2=0.99, beta3=0.999,
                 weight_decay=0.1, lambda_b=0.1, eps=1e-8, gamma=5.0):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, beta3=beta3,
                       weight_decay=weight_decay, lambda_b=lambda_b, eps=eps, gamma=gamma)
        super().__init__(params, defaults)
        self._init_sqrt_dim()

    def _init_sqrt_dim(self):
        total = sum(p.numel() for g in self.param_groups for p in g["params"])
        self.sqrt_dim = math.sqrt(total)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        all_smooth_pre, all_belief, all_params = [], [], []

        for group in self.param_groups:
            eps, gamma = group["eps"], group["gamma"]
            beta1, beta2, beta3 = group["beta1"], group["beta2"], group["beta3"]
            lambda_b = group["lambda_b"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state["m"] = torch.zeros_like(p)
                    state["s"] = torch.zeros_like(p)

                m, s = state["m"], state["s"]
                s.mul_(beta3).addcmul_(g, g, value=(1.0 - beta3))
                c = beta1 * m + (1.0 - beta1) * g
                smooth = torch.tanh(gamma * c)
                smooth_pre = smooth / (s.sqrt() + eps)
                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                belief = lambda_b * (delta / delta_norm)

                all_smooth_pre.append(smooth_pre)
                all_belief.append(belief)
                all_params.append((p, group))

        if not all_params:
            return loss

        s_norm = sum((sp * sp).sum() for sp in all_smooth_pre).sqrt().clamp(min=1e-8)
        scale = self.sqrt_dim / s_norm

        for (p, group), sp, b in zip(all_params, all_smooth_pre, all_belief):
            lr, wd, beta2 = group["lr"], group["weight_decay"], group["beta2"]
            d = scale * sp + b
            state = self.state[p]
            if wd != 0.0:
                p.mul_(1.0 - lr * wd)
            p.add_(d, alpha=-lr)
            state["m"].mul_(beta2).add_(p.grad, alpha=(1.0 - beta2))

        return loss


class SmoothLiftedRLO(Optimizer):
    """Second-order lifted RLO."""
    def __init__(self, params, lr=1e-4, beta1=0.9, beta2=0.99, eta=0.3,
                 weight_decay=0.1, lambda_b=0.1, eps=1e-8, gamma=5.0):
        defaults = dict(lr=lr, beta1=beta1, beta2=beta2, eta=eta,
                       weight_decay=weight_decay, lambda_b=lambda_b, eps=eps, gamma=gamma)
        super().__init__(params, defaults)
        self._init_sqrt_dim()

    def _init_sqrt_dim(self):
        total = sum(p.numel() for g in self.param_groups for p in g["params"])
        self.sqrt_dim = math.sqrt(total)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        all_s, all_b, all_params = [], [], []

        for group in self.param_groups:
            eps, gamma = group["eps"], group["gamma"]
            beta1, lambda_b = group["beta1"], group["lambda_b"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state["m"] = torch.zeros_like(p)
                    state["v"] = torch.zeros_like(p)

                m = state["m"]
                c = beta1 * m + (1.0 - beta1) * g
                s = torch.tanh(gamma * c)
                delta = g - m
                delta_norm = delta.norm(p=2).clamp(min=eps)
                b = lambda_b * (delta / delta_norm)

                all_s.append(s)
                all_b.append(b)
                all_params.append((p, group))

        if not all_params:
            return loss

        s_norm = sum((s * s).sum() for s in all_s).sqrt().clamp(min=1e-8)
        scale = self.sqrt_dim / s_norm

        for (p, group), s, b in zip(all_params, all_s, all_b):
            lr, wd, eta, beta2 = group["lr"], group["weight_decay"], group["eta"], group["beta2"]
            d = scale * s + b
            state = self.state[p]
            m, v = state["m"], state["v"]
            if wd != 0.0:
                p.mul_(1.0 - lr * wd)
            v.mul_(1.0 - eta).add_(d, alpha=eta)
            p.add_(v, alpha=-lr)
            m.mul_(beta2).add_(p.grad, alpha=(1.0 - beta2))

        return loss


class Lion(Optimizer):
    """Lion optimizer (Google)."""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr, wd = group['lr'], group['weight_decay']
            beta1, beta2 = group['betas']
            for p in group['params']:
                if p.grad is None:
                    continue
                if wd != 0.0:
                    p.mul_(1.0 - lr * wd)
                g = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)
                m = state['exp_avg']
                update = (beta1 * m + (1 - beta1) * g).sign()
                p.add_(update, alpha=-lr)
                m.mul_(beta2).add_(g, alpha=1 - beta2)

        return loss


OPTIMIZERS = {
    'adamw': lambda p, lr, wd: torch.optim.AdamW(p, lr=lr, weight_decay=wd),
    'lion': lambda p, lr, wd: Lion(p, lr=lr*0.1, weight_decay=wd*10),
    'rlo': lambda p, lr, wd: RLO(p, lr=lr*0.1, weight_decay=wd*10),
    'rlo_lambda_a': lambda p, lr, wd: RLO_LambdaA(p, lr=lr*0.1, weight_decay=wd*10),
    'smooth_lifted_rlo': lambda p, lr, wd: SmoothLiftedRLO(p, lr=lr*0.1, weight_decay=wd*10),
}

print("Optimizers loaded:", list(OPTIMIZERS.keys()))

In [ ]:
# =============================================================================
# CELL 5: Data Loading (ImageFolder - No HuggingFace)
# =============================================================================

def get_imagenet_loaders(
    root: Path,
    batch_size: int = 256,
    num_workers: int = 8,
    image_size: int = 224,
    use_randaugment: bool = True,
):
    """Load ImageNet using ImageFolder (fast, no hanging)."""
    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    
    train_transforms = [
        T.RandomResizedCrop(image_size, interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
    ]
    if use_randaugment:
        train_transforms.append(T.RandAugment(num_ops=2, magnitude=9))
    train_transforms.extend([T.ToTensor(), T.Normalize(MEAN, STD)])
    
    val_transform = T.Compose([
        T.Resize(int(image_size * 256 / 224), interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(image_size),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])
    
    train_dataset = ImageFolder(root / 'train', T.Compose(train_transforms))
    val_dataset = ImageFolder(root / 'val', val_transform)
    
    print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Classes: {len(train_dataset.classes)}")
    
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=num_workers, pin_memory=True, drop_last=True,
        persistent_workers=num_workers > 0, prefetch_factor=4 if num_workers > 0 else None,
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size * 2, shuffle=False,
        num_workers=num_workers, pin_memory=True,
        persistent_workers=num_workers > 0,
    )
    
    return train_loader, val_loader


class GPUMixup:
    """GPU Mixup/CutMix."""
    def __init__(self, mixup_alpha=0.8, cutmix_alpha=1.0, prob=1.0, switch_prob=0.5, num_classes=1000):
        self.mixup_alpha, self.cutmix_alpha = mixup_alpha, cutmix_alpha
        self.prob, self.switch_prob, self.num_classes = prob, switch_prob, num_classes
        
    @torch.no_grad()
    def __call__(self, x, target):
        if random.random() > self.prob:
            return x, F.one_hot(target, self.num_classes).float()
        use_cutmix = random.random() < self.switch_prob
        alpha = self.cutmix_alpha if use_cutmix else self.mixup_alpha
        lam = np.random.beta(alpha, alpha)
        B = x.size(0)
        index = torch.randperm(B, device=x.device)
        if use_cutmix:
            _, _, H, W = x.shape
            cut_rat = math.sqrt(1.0 - lam)
            cut_w, cut_h = int(W * cut_rat), int(H * cut_rat)
            cx, cy = random.randint(0, W), random.randint(0, H)
            x1, y1 = max(cx - cut_w//2, 0), max(cy - cut_h//2, 0)
            x2, y2 = min(cx + cut_w//2, W), min(cy + cut_h//2, H)
            x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
            lam = 1 - ((x2-x1)*(y2-y1) / (W*H))
        else:
            x = lam * x + (1-lam) * x[index]
        y = F.one_hot(target, self.num_classes).float()
        y_perm = F.one_hot(target[index], self.num_classes).float()
        return x, lam * y + (1-lam) * y_perm


print("Data loading ready.")

In [ ]:
# =============================================================================
# CELL 6: Models
# =============================================================================

def create_resnet50(num_classes=1000):
    from torchvision.models import resnet50
    return resnet50(weights=None, num_classes=num_classes)


class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads, self.head_dim = num_heads, dim // num_heads
        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = attn_drop
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)
    
    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)
        x = F.scaled_dot_product_attention(q, k, v, dropout_p=self.attn_drop if self.training else 0.0)
        return self.proj_drop(self.proj(x.transpose(1, 2).reshape(B, N, C)))


class Block(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4., qkv_bias=True, drop=0., attn_drop=0.):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = Attention(dim, num_heads, qkv_bias, attn_drop, drop)
        self.norm2 = nn.LayerNorm(dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(nn.Linear(dim, hidden), nn.GELU(), nn.Dropout(drop),
                                  nn.Linear(hidden, dim), nn.Dropout(drop))
    
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))


class VisionTransformer(nn.Module):
    def __init__(self, img_size=224, patch_size=16, in_chans=3, num_classes=1000,
                 embed_dim=768, depth=12, num_heads=12, mlp_ratio=4., drop_rate=0.):
        super().__init__()
        num_patches = (img_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        self.pos_drop = nn.Dropout(p=drop_rate)
        self.blocks = nn.ModuleList([Block(embed_dim, num_heads, mlp_ratio, drop=drop_rate) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
    
    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = torch.cat((self.cls_token.expand(B, -1, -1), x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        return self.head(self.norm(x)[:, 0])


def create_vit_small(num_classes=1000):
    return VisionTransformer(embed_dim=384, depth=12, num_heads=6, num_classes=num_classes)

def create_vit_base(num_classes=1000):
    return VisionTransformer(embed_dim=768, depth=12, num_heads=12, num_classes=num_classes)

def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"ResNet-50: {count_params(create_resnet50())/1e6:.1f}M")
print(f"ViT-S/16: {count_params(create_vit_small())/1e6:.1f}M")
print(f"ViT-B/16: {count_params(create_vit_base())/1e6:.1f}M")

In [ ]:
# =============================================================================
# CELL 7: Training Engine with Plotting
# =============================================================================

class CosineScheduler:
    def __init__(self, optimizer, base_lr, total_steps, warmup_steps, min_lr=0):
        self.optimizer, self.base_lr = optimizer, base_lr
        self.total_steps, self.warmup_steps, self.min_lr = total_steps, warmup_steps, min_lr
        self.step_count = 0
        
    def step(self):
        self.step_count += 1
        lr = self._get_lr()
        for g in self.optimizer.param_groups:
            g['lr'] = lr
        return lr
    
    def _get_lr(self):
        if self.step_count < self.warmup_steps:
            return self.base_lr * self.step_count / max(1, self.warmup_steps)
        progress = (self.step_count - self.warmup_steps) / max(1, self.total_steps - self.warmup_steps)
        return self.min_lr + 0.5 * (self.base_lr - self.min_lr) * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits = model(images)
            loss = criterion(logits, labels)
        total_loss += loss.item() * images.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / max(total, 1), 100.0 * correct / max(total, 1)


def plot_experiment_results(results: Dict, title: str, save_path: Path = None):
    """Plot training curves for all optimizers in an experiment."""
    colors = {'adamw': '#1f77b4', 'lion': '#ff7f0e', 'rlo': '#2ca02c',
              'rlo_lambda_a': '#d62728', 'smooth_lifted_rlo': '#9467bd'}
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    for name, res in results.items():
        h = res.get('history', {})
        opt = name.split('_')[-1] if name.count('_') > 1 else name
        if opt not in colors:
            for k in colors:
                if k in name:
                    opt = k
                    break
        c = colors.get(opt, 'gray')
        label = name
        
        if 'train_acc' in h:
            axes[0].plot(h['train_acc'], label=label, color=c, linewidth=2)
        if 'val_acc' in h:
            axes[1].plot(h['val_acc'], label=label, color=c, linewidth=2)
        if 'throughput' in h:
            axes[2].plot(h['throughput'], label=label, color=c, linewidth=2)
    
    axes[0].set_title('Training Accuracy', fontsize=14)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].set_title('Validation Accuracy', fontsize=14)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    axes[2].set_title('Throughput', fontsize=14)
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Images/sec')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    
    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Plot saved to {save_path}")
    
    plt.show()


def print_results_table(results: Dict, title: str):
    """Print results as a formatted table."""
    print(f"\n{'='*70}")
    print(f"{title}")
    print(f"{'='*70}")
    print(f"{'Optimizer':<25} {'Best Acc':>10} {'Final Acc':>10} {'Throughput':>15}")
    print(f"{'-'*60}")
    
    sorted_results = sorted(results.items(), key=lambda x: x[1].get('best_acc', 0), reverse=True)
    for name, res in sorted_results:
        best = res.get('best_acc', 0)
        final = res.get('final_acc', 0)
        thr = res.get('avg_throughput', 0)
        print(f"{name:<25} {best:>9.2f}% {final:>9.2f}% {thr:>12.0f} img/s")


print("Training engine ready.")

In [ ]:
# =============================================================================
# CELL 8: Main Training Function
# =============================================================================

def train_model(
    model_name: str,
    opt_name: str,
    epochs: int = 90,
    batch_size: int = 2048,
    lr: float = 1e-3,
    wd: float = 0.05,
    warmup_epochs: int = 5,
    use_randaugment: bool = False,
    use_mixup: bool = False,
    use_compile: bool = True,
    label_smoothing: float = 0.1,
    gpu_ids: List[int] = None,
):
    """Train a model with DataParallel on multiple GPUs."""
    set_seed(42)
    
    if gpu_ids is None:
        gpu_ids = list(range(torch.cuda.device_count()))
    num_gpus = len(gpu_ids)
    device = torch.device(f"cuda:{gpu_ids[0]}")
    
    run_name = f"{model_name}_{opt_name}"
    
    print(f"\n{'='*70}")
    print(f"Training: {run_name}")
    print(f"GPUs: {gpu_ids}, Batch: {batch_size}, Epochs: {epochs}")
    print(f"{'='*70}")
    
    # Data
    train_loader, val_loader = get_imagenet_loaders(
        IMAGENET_FOLDER, batch_size=batch_size,
        num_workers=8 * num_gpus, use_randaugment=use_randaugment,
    )
    
    # Model
    if model_name == "resnet50":
        model = create_resnet50()
    elif model_name == "vit_s16":
        model = create_vit_small()
    elif model_name == "vit_b16":
        model = create_vit_base()
    else:
        raise ValueError(f"Unknown model: {model_name}")
    
    model = model.to(device)
    
    if use_compile:
        print("Compiling model...")
        model = torch.compile(model, mode='reduce-overhead')
    
    if num_gpus > 1:
        model = nn.DataParallel(model, device_ids=gpu_ids)
    
    print(f"Params: {count_params(model)/1e6:.1f}M")
    
    # Optimizer
    optimizer = OPTIMIZERS[opt_name](model.parameters(), lr, wd)
    actual_lr = optimizer.param_groups[0]['lr']
    print(f"Optimizer: {opt_name}, lr={actual_lr:.2e}")
    
    # Scheduler
    total_steps = len(train_loader) * epochs
    warmup_steps = len(train_loader) * warmup_epochs
    scheduler = CosineScheduler(optimizer, actual_lr, total_steps, warmup_steps)
    
    # Loss
    criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    mixup = GPUMixup() if use_mixup else None
    
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': [], 'throughput': [], 'lr': []}
    best_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        epoch_start = time.time()
        correct, total, running_loss = 0, 0, 0.0
        
        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
        for step, (images, labels) in enumerate(pbar):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            
            if mixup:
                images, labels_mixed = mixup(images, labels)
                use_soft = True
            else:
                labels_mixed = None
                use_soft = False
            
            optimizer.zero_grad(set_to_none=True)
            
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits = model(images)
                if use_soft:
                    loss = -torch.sum(F.log_softmax(logits, 1) * labels_mixed, 1).mean()
                else:
                    loss = criterion(logits, labels)
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            pred = logits.argmax(1)
            correct += (pred == labels).sum().item()
            total += labels.size(0)
            running_loss += loss.item()
            
            if step % 50 == 0:
                pbar.set_postfix({'loss': f'{running_loss/(step+1):.4f}', 'acc': f'{100*correct/total:.1f}%'})
        
        epoch_time = time.time() - epoch_start
        throughput = len(train_loader) * batch_size / epoch_time
        train_acc = 100 * correct / total
        train_loss = running_loss / len(train_loader)
        
        # Validation
        model_eval = model.module if hasattr(model, 'module') else model
        val_loss, val_acc = evaluate(model_eval, val_loader, criterion, device)
        
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['throughput'].append(throughput)
        history['lr'].append(scheduler._get_lr())
        
        is_best = val_acc > best_acc
        best_acc = max(val_acc, best_acc)
        
        print(f"Epoch {epoch+1}: Train={train_acc:.1f}%, Val={val_acc:.1f}% {'*' if is_best else ''}, "
              f"Throughput={throughput:.0f} img/s")
        
        # Save checkpoint
        if is_best:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model_eval.state_dict(),
                'best_acc': best_acc,
            }, RESULTS_DIR / f"{run_name}_best.pt")
    
    result = {
        'run_name': run_name,
        'model': model_name,
        'optimizer': opt_name,
        'best_acc': best_acc,
        'final_acc': history['val_acc'][-1],
        'avg_throughput': np.mean(history['throughput']),
        'history': history,
        'config': {
            'epochs': epochs, 'batch_size': batch_size, 'lr': lr, 'wd': wd,
            'warmup_epochs': warmup_epochs, 'use_randaugment': use_randaugment,
            'use_mixup': use_mixup, 'label_smoothing': label_smoothing,
        }
    }
    
    with open(RESULTS_DIR / f"{run_name}.json", 'w') as f:
        json.dump(result, f, indent=2, default=str)
    
    return result


print("Main training function ready.")

In [ ]:
# =============================================================================
# CELL 9: Experiment 4.1 - ImageNet Classification
# =============================================================================

def run_experiment_4_1_resnet50():
    """
    Experiment 4.1.1: ResNet-50 on ImageNet
    - 90 epochs
    - Batch size: 4096 (512 per GPU x 8)
    - No RandAugment, No Mixup (following LION paper for ResNet)
    """
    print("\n" + "#"*70)
    print("# EXPERIMENT 4.1.1: ResNet-50 on ImageNet")
    print("#"*70)
    
    results = {}
    optimizers = ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']
    
    for opt in optimizers:
        try:
            result = train_model(
                model_name="resnet50",
                opt_name=opt,
                epochs=90,
                batch_size=4096,
                lr=0.4,  # Scaled for large batch: 0.1 * (4096/1024)
                wd=1e-4,
                warmup_epochs=5,
                use_randaugment=False,
                use_mixup=False,
                label_smoothing=0.0,
                use_compile=True,
            )
            results[f"resnet50_{opt}"] = result
        except Exception as e:
            print(f"Failed {opt}: {e}")
            import traceback
            traceback.print_exc()
    
    # Plot and save
    print_results_table(results, "ResNet-50 Results")
    plot_experiment_results(results, "Experiment 4.1.1: ResNet-50 on ImageNet",
                           RESULTS_DIR / "exp_4_1_1_resnet50.png")
    
    with open(RESULTS_DIR / "exp_4_1_1_resnet50.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    return results


def run_experiment_4_1_vit_s16():
    """
    Experiment 4.1.2: ViT-S/16 on ImageNet
    - 300 epochs
    - Batch size: 2048 (256 per GPU x 8)
    - With RandAugment + Mixup (following LION paper for ViT)
    """
    print("\n" + "#"*70)
    print("# EXPERIMENT 4.1.2: ViT-S/16 on ImageNet")
    print("#"*70)
    
    results = {}
    optimizers = ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']
    
    for opt in optimizers:
        try:
            result = train_model(
                model_name="vit_s16",
                opt_name=opt,
                epochs=300,
                batch_size=2048,
                lr=1e-3,
                wd=0.05,
                warmup_epochs=30,
                use_randaugment=True,
                use_mixup=True,
                label_smoothing=0.1,
                use_compile=True,
            )
            results[f"vit_s16_{opt}"] = result
        except Exception as e:
            print(f"Failed {opt}: {e}")
    
    print_results_table(results, "ViT-S/16 Results")
    plot_experiment_results(results, "Experiment 4.1.2: ViT-S/16 on ImageNet",
                           RESULTS_DIR / "exp_4_1_2_vit_s16.png")
    
    with open(RESULTS_DIR / "exp_4_1_2_vit_s16.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    return results


def run_experiment_4_1_vit_b16():
    """
    Experiment 4.1.3: ViT-B/16 on ImageNet
    - 300 epochs
    - Batch size: 1024 (128 per GPU x 8)
    """
    print("\n" + "#"*70)
    print("# EXPERIMENT 4.1.3: ViT-B/16 on ImageNet")
    print("#"*70)
    
    results = {}
    optimizers = ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']
    
    for opt in optimizers:
        try:
            result = train_model(
                model_name="vit_b16",
                opt_name=opt,
                epochs=300,
                batch_size=1024,
                lr=1e-3,
                wd=0.05,
                warmup_epochs=30,
                use_randaugment=True,
                use_mixup=True,
                label_smoothing=0.1,
                use_compile=True,
            )
            results[f"vit_b16_{opt}"] = result
        except Exception as e:
            print(f"Failed {opt}: {e}")
    
    print_results_table(results, "ViT-B/16 Results")
    plot_experiment_results(results, "Experiment 4.1.3: ViT-B/16 on ImageNet",
                           RESULTS_DIR / "exp_4_1_3_vit_b16.png")
    
    with open(RESULTS_DIR / "exp_4_1_3_vit_b16.json", 'w') as f:
        json.dump(results, f, indent=2, default=str)
    
    return results

In [ ]:
# =============================================================================
# CELL 10: Experiment 4.3 - Diffusion Models
# =============================================================================

class SimpleUNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=128):
        super().__init__()
        def block(ic, oc):
            return nn.Sequential(
                nn.Conv2d(ic, oc, 3, padding=1), nn.GroupNorm(8, oc), nn.SiLU(),
                nn.Conv2d(oc, oc, 3, padding=1), nn.GroupNorm(8, oc), nn.SiLU(),
            )
        self.enc1, self.enc2, self.enc3 = block(in_ch, base_ch), block(base_ch, base_ch*2), block(base_ch*2, base_ch*4)
        self.mid = block(base_ch*4, base_ch*4)
        self.dec3, self.dec2, self.dec1 = block(base_ch*8, base_ch*2), block(base_ch*4, base_ch), block(base_ch*2, base_ch)
        self.final = nn.Conv2d(base_ch, in_ch, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
    
    def forward(self, x, t):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        m = self.mid(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up(m), e3], 1))
        d2 = self.dec2(torch.cat([self.up(d3), e2], 1))
        d1 = self.dec1(torch.cat([self.up(d2), e1], 1))
        return self.final(d1)


def run_experiment_4_3_diffusion():
    """Experiment 4.3: Diffusion Models on CIFAR-10."""
    print("\n" + "#"*70)
    print("# EXPERIMENT 4.3: Diffusion Models")
    print("#"*70)
    
    device = torch.device("cuda:0")
    
    transform = T.Compose([T.Resize(64), T.RandomHorizontalFlip(), T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
    dataset = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=transform)
    loader = DataLoader(dataset, batch_size=512, shuffle=True, num_workers=8, pin_memory=True, drop_last=True)
    
    T_steps = 1000
    betas = torch.linspace(0.0001, 0.02, T_steps, device=device)
    alpha_bar = torch.cumprod(1.0 - betas, 0)
    sqrt_ab = torch.sqrt(alpha_bar)
    sqrt_1mab = torch.sqrt(1.0 - alpha_bar)
    
    results = {}
    
    for opt_name in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        print(f"\n--- {opt_name} ---")
        set_seed(42)
        
        model = SimpleUNet().to(device)
        model = torch.compile(model, mode='reduce-overhead')
        model = nn.DataParallel(model)
        
        optimizer = OPTIMIZERS[opt_name](model.parameters(), 3e-4, 0.01)
        history = {'loss': []}
        
        epochs = 50
        for epoch in range(epochs):
            model.train()
            losses = []
            for imgs, _ in tqdm(loader, desc=f"Epoch {epoch+1}", leave=False):
                imgs = imgs.to(device, non_blocking=True)
                t = torch.randint(0, T_steps, (imgs.size(0),), device=device)
                noise = torch.randn_like(imgs)
                noisy = sqrt_ab[t].view(-1,1,1,1) * imgs + sqrt_1mab[t].view(-1,1,1,1) * noise
                
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    pred = model(noisy, t.float() / T_steps)
                    loss = F.mse_loss(pred, noise)
                loss.backward()
                optimizer.step()
                losses.append(loss.item())
            
            avg = np.mean(losses)
            history['loss'].append(avg)
            if (epoch + 1) % 10 == 0:
                print(f"  Epoch {epoch+1}: Loss={avg:.4f}")
        
        results[opt_name] = {'final_loss': history['loss'][-1], 'history': history}
    
    # Plot
    plt.figure(figsize=(10, 6))
    colors = {'adamw': '#1f77b4', 'lion': '#ff7f0e', 'rlo': '#2ca02c',
              'rlo_lambda_a': '#d62728', 'smooth_lifted_rlo': '#9467bd'}
    for name, res in results.items():
        plt.plot(res['history']['loss'], label=name, color=colors[name], linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.title('Experiment 4.3: Diffusion Models on CIFAR-10')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(RESULTS_DIR / "exp_4_3_diffusion.png", dpi=150)
    plt.show()
    
    print("\nDiffusion Results:")
    for name, res in sorted(results.items(), key=lambda x: x[1]['final_loss']):
        print(f"  {name}: {res['final_loss']:.4f}")
    
    return results

In [ ]:
# =============================================================================
# CELL 11: Experiment 4.4 - Language Modeling
# =============================================================================

class TransformerLM(nn.Module):
    def __init__(self, vocab_size=32000, d_model=512, n_heads=8, n_layers=6, max_len=512):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = nn.Parameter(torch.zeros(1, max_len, d_model))
        layer = nn.TransformerEncoderLayer(d_model, n_heads, d_model*4, dropout=0.1, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers)
        self.ln = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.embed.weight
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.vocab_size = vocab_size
    
    def forward(self, x):
        B, T = x.shape
        x = self.embed(x) * math.sqrt(self.embed.embedding_dim) + self.pos[:, :T]
        mask = torch.triu(torch.ones(T, T, device=x.device), 1).bool()
        x = self.transformer(x, mask=mask, is_causal=True)
        return self.head(self.ln(x))


def run_experiment_4_4_lm():
    """Experiment 4.4: Language Modeling."""
    print("\n" + "#"*70)
    print("# EXPERIMENT 4.4: Language Modeling")
    print("#"*70)
    
    device = torch.device("cuda:0")
    vocab_size = 32000
    seq_len = 512
    batch_size = 32
    
    # Synthetic data (for real experiments, use OpenWebText or similar)
    data = torch.randint(0, vocab_size, (500000,))
    
    def get_batch():
        ix = torch.randint(len(data) - seq_len, (batch_size,))
        x = torch.stack([data[i:i+seq_len] for i in ix]).to(device)
        y = torch.stack([data[i+1:i+seq_len+1] for i in ix]).to(device)
        return x, y
    
    results = {}
    
    for opt_name in ['adamw', 'lion', 'rlo', 'rlo_lambda_a', 'smooth_lifted_rlo']:
        print(f"\n--- {opt_name} ---")
        set_seed(42)
        
        model = TransformerLM(vocab_size, d_model=768, n_heads=12, n_layers=12).to(device)
        model = torch.compile(model, mode='reduce-overhead')
        model = nn.DataParallel(model)
        
        optimizer = OPTIMIZERS[opt_name](model.parameters(), 3e-4, 0.1)
        criterion = nn.CrossEntropyLoss()
        history = {'ppl': []}
        
        epochs = 20
        steps_per_epoch = 500
        
        for epoch in range(epochs):
            model.train()
            losses = []
            for _ in tqdm(range(steps_per_epoch), desc=f"Epoch {epoch+1}", leave=False):
                x, y = get_batch()
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                    logits = model(x)
                    loss = criterion(logits.view(-1, vocab_size), y.view(-1))
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                losses.append(loss.item())
            
            ppl = math.exp(np.mean(losses))
            history['ppl'].append(ppl)
            print(f"  Epoch {epoch+1}: PPL={ppl:.2f}")
        
        results[opt_name] = {'final_ppl': history['ppl'][-1], 'history': history}
    
    # Plot
    plt.figure(figsize=(10, 6))
    colors = {'adamw': '#1f77b4', 'lion': '#ff7f0e', 'rlo': '#2ca02c',
              'rlo_lambda_a': '#d62728', 'smooth_lifted_rlo': '#9467bd'}
    for name, res in results.items():
        plt.plot(res['history']['ppl'], label=name, color=colors[name], linewidth=2)
    plt.xlabel('Epoch')
    plt.ylabel('Perplexity')
    plt.title('Experiment 4.4: Language Modeling')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.savefig(RESULTS_DIR / "exp_4_4_lm.png", dpi=150)
    plt.show()
    
    print("\nLanguage Modeling Results:")
    for name, res in sorted(results.items(), key=lambda x: x[1]['final_ppl']):
        print(f"  {name}: PPL={res['final_ppl']:.2f}")
    
    return results

In [ ]:
# =============================================================================
# CELL 12: Quick Test
# =============================================================================

def quick_test(epochs=3, batch_size=2048):
    """Quick test to verify everything works."""
    print("\n" + "="*70)
    print("QUICK TEST: ResNet-50 + AdamW (3 epochs)")
    print("="*70)
    
    result = train_model(
        model_name="resnet50",
        opt_name="adamw",
        epochs=epochs,
        batch_size=batch_size,
        lr=0.4,
        wd=1e-4,
        warmup_epochs=1,
        use_compile=True,
    )
    
    print(f"\n✓ Quick test complete!")
    print(f"  Best acc: {result['best_acc']:.2f}%")
    print(f"  Throughput: {result['avg_throughput']:.0f} img/s")
    
    return result

# Run quick test
# quick_result = quick_test()

In [ ]:
# =============================================================================
# CELL 13: Run All Experiments
# =============================================================================

def run_all_experiments():
    """Run all experiments from LION paper."""
    all_results = {}
    
    # 4.1.1 ResNet-50
    print("\n" + "="*80)
    print("Starting Experiment 4.1.1: ResNet-50")
    print("="*80)
    all_results['resnet50'] = run_experiment_4_1_resnet50()
    
    # 4.1.2 ViT-S/16
    print("\n" + "="*80)
    print("Starting Experiment 4.1.2: ViT-S/16")
    print("="*80)
    all_results['vit_s16'] = run_experiment_4_1_vit_s16()
    
    # 4.1.3 ViT-B/16
    print("\n" + "="*80)
    print("Starting Experiment 4.1.3: ViT-B/16")
    print("="*80)
    all_results['vit_b16'] = run_experiment_4_1_vit_b16()
    
    # 4.3 Diffusion
    print("\n" + "="*80)
    print("Starting Experiment 4.3: Diffusion")
    print("="*80)
    all_results['diffusion'] = run_experiment_4_3_diffusion()
    
    # 4.4 Language Modeling
    print("\n" + "="*80)
    print("Starting Experiment 4.4: Language Modeling")
    print("="*80)
    all_results['lm'] = run_experiment_4_4_lm()
    
    # Save all results
    with open(RESULTS_DIR / "all_experiments.json", 'w') as f:
        json.dump(all_results, f, indent=2, default=str)
    
    print("\n" + "="*80)
    print("ALL EXPERIMENTS COMPLETE!")
    print("="*80)
    
    return all_results


# Uncomment to run all:
# all_results = run_all_experiments()

---

# Experiment Analysis

## Required Experiments (Following LION Paper Section 4)

### 4.1 Image Classification on ImageNet

| Model | Epochs | Batch Size | Augmentation | Status |
|-------|--------|------------|--------------|--------|
| ResNet-50 | 90 | 4096 | None | ⬜ |
| ViT-S/16 | 300 | 2048 | RandAug + Mixup | ⬜ |
| ViT-B/16 | 300 | 1024 | RandAug + Mixup | ⬜ |

**Key Metrics:**
- Top-1 Accuracy
- Training throughput (img/s)
- Memory usage

**Expected Results (from LION paper):**
- ResNet-50: AdamW ~76.5%, LION ~77.0%
- ViT-S/16: AdamW ~79.2%, LION ~79.7%
- ViT-B/16: AdamW ~81.4%, LION ~81.8%

### 4.3 Diffusion Models

| Dataset | Model | Epochs | Status |
|---------|-------|--------|--------|
| CIFAR-10 | U-Net | 50 | ⬜ |

**Key Metrics:**
- MSE Loss
- FID Score (optional)

### 4.4 Language Modeling

| Model | Epochs | Status |
|-------|--------|--------|
| GPT-style Transformer | 20 | ⬜ |

**Key Metrics:**
- Perplexity (PPL)

---

## Additional Experiments (For Paper Strength)

### Ablation Studies

1. **Belief Coefficient Sensitivity**
   - Test belief_coef = [0.0, 0.05, 0.1, 0.2, 0.5]
   - belief_coef=0.0 should reduce RLO to LION

2. **Gamma Sensitivity** (for RLO_LambdaA and SmoothLiftedRLO)
   - Test gamma = [1.0, 3.0, 5.0, 10.0]

3. **Eta Sensitivity** (for SmoothLiftedRLO)
   - Test eta = [0.1, 0.3, 0.5, 0.7]

### Learning Rate Sensitivity

- Test lr_scale = [0.03, 0.1, 0.3, 1.0] relative to AdamW
- Currently using 0.1x for sign-based optimizers

### Convergence Analysis

- Plot training loss curves (not just accuracy)
- Gradient norm over training
- Update magnitude statistics

---

## How to Run

```python
# Step 1: Convert ImageNet (only once)
if NEED_CONVERSION:
    convert_hf_to_imagefolder(HF_CACHE_PATH, IMAGENET_FOLDER)

# Step 2: Quick test
quick_result = quick_test()

# Step 3: Run all experiments
all_results = run_all_experiments()
```

---

## Expected Time

With 8x B200:

| Experiment | Estimated Time |
|------------|----------------|
| ResNet-50 (90 epochs × 5 optimizers) | ~15 hours |
| ViT-S/16 (300 epochs × 5 optimizers) | ~40 hours |
| ViT-B/16 (300 epochs × 5 optimizers) | ~60 hours |
| Diffusion (50 epochs × 5 optimizers) | ~5 hours |
| Language Model (20 epochs × 5 optimizers) | ~3 hours |
| **Total** | **~120 hours (5 days)** |

In [ ]:
# =============================================================================
# CELL 14: Summary
# =============================================================================

print("""
================================================================================
RLO EXPERIMENTS NOTEBOOK - READY
================================================================================

Hardware: 8x NVIDIA B200 (183GB each)

Steps to run:

1. CONVERT DATA (Cell 3) - Only needed once!
   if NEED_CONVERSION:
       convert_hf_to_imagefolder(HF_CACHE_PATH, IMAGENET_FOLDER)

2. QUICK TEST (Cell 12)
   quick_result = quick_test()

3. RUN EXPERIMENTS
   Option A: Run all at once
       all_results = run_all_experiments()
   
   Option B: Run individually
       resnet_results = run_experiment_4_1_resnet50()
       vit_s_results = run_experiment_4_1_vit_s16()
       vit_b_results = run_experiment_4_1_vit_b16()
       diffusion_results = run_experiment_4_3_diffusion()
       lm_results = run_experiment_4_4_lm()

Results saved to: ./rlo_results/

Expected throughput:
  - ResNet-50: ~15,000-20,000 img/s
  - ViT-S/16: ~8,000-12,000 img/s
  - ViT-B/16: ~4,000-6,000 img/s

================================================================================
""")